In [1]:
import pandas as pd
import s3fs

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("🏆 Construyendo indicadores estratégicos para la Capa Gold...")

# 1. Asegurar que tenemos nuestra tabla maestra integrada en memoria (df_master)
# Si no la tienes cargada en este instante, la unimos rápidamente desde las tablas transformadas:
df_trans = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_transaction_transformed.parquet", storage_options=storage_opts)
df_id = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_identity_transformed.parquet", storage_options=storage_opts)
df_master = pd.merge(df_trans, df_id, on="TransactionID", how="left")

# ==============================================================================
# TABLA GOLD 1: Matriz de Riesgo por Franja Horaria y Día de la Semana
# ==============================================================================
gold_tiempo = df_master.groupby(['dia_semana', 'franja_horaria'], as_index=False).agg(
    total_tx=('TransactionID', 'count'),
    fraudes=('isFraud', 'sum'),
    monto_expuesto_pen=('monto_pen', 'sum')
)
gold_tiempo['tasa_fraude_pct'] = (gold_tiempo['fraudes'] / gold_tiempo['total_tx'] * 100).round(2)

# Guardar en Capa Gold MinIO
gold_tiempo.to_parquet(f"s3://{BUCKET_NAME}/gold/kpis_riesgo_temporal.parquet", index=False, storage_options=storage_opts)


# ==============================================================================
# TABLA GOLD 2: Perfil de Riesgo por Dispositivo y Sistema Operativo
# ==============================================================================
gold_dispositivos = df_master.groupby(['DeviceType', 'familia_os'], as_index=False).agg(
    total_tx=('TransactionID', 'count'),
    fraudes=('isFraud', 'sum'),
    monto_fraude_pen=('monto_pen', lambda x: x[df_master.loc[x.index, 'isFraud'] == 1].sum())
)
gold_dispositivos['tasa_fraude_pct'] = (gold_dispositivos['fraudes'] / gold_dispositivos['total_tx'] * 100).round(2)

# Guardar en Capa Gold MinIO
gold_dispositivos.to_parquet(f"s3://{BUCKET_NAME}/gold/kpis_riesgo_dispositivos.parquet", index=False, storage_options=storage_opts)

print("✨ ¡Tablas de la Capa Gold generadas y almacenadas con éxito en MinIO!")

print("\n📊 Vista previa - KPIs de Riesgo Temporal:")
display(gold_tiempo.sort_values(by='tasa_fraude_pct', ascending=False).head(5))

print("\n📊 Vista previa - KPIs por Dispositivo y OS:")
display(gold_dispositivos.sort_values(by='tasa_fraude_pct', ascending=False))

🏆 Construyendo indicadores estratégicos para la Capa Gold...
✨ ¡Tablas de la Capa Gold generadas y almacenadas con éxito en MinIO!

📊 Vista previa - KPIs de Riesgo Temporal:


,dia_semana,franja_horaria,total_tx,fraudes,monto_expuesto_pen,tasa_fraude_pct
13,Sunday,Mañana (06-11h),2597,278,1118548.875,10.70
9,Saturday,Mañana (06-11h),2881,269,1327162.875,9.34
1,Friday,Mañana (06-11h),4533,344,2199797.250,7.59
17,Thursday,Mañana (06-11h),4025,302,1948575.000,7.50
5,Monday,Mañana (06-11h),3800,227,1530656.000,5.97



📊 Vista previa - KPIs por Dispositivo y OS:


,DeviceType,familia_os,total_tx,fraudes,monto_fraude_pen,tasa_fraude_pct
8,mobile,Linux,20,4,1.500000e+02,20.00
10,mobile,Otro/Desconocido,29517,3871,6.595295e+05,13.11
4,desktop,Otro/Desconocido,33753,3907,8.486956e+05,11.58
7,mobile,Android,6302,542,3.787350e+05,8.60
2,desktop,Linux,1116,81,4.253250e+04,7.26
12,mobile,iOS,19781,1240,1.010689e+06,6.27
5,desktop,Windows,36722,1268,6.240262e+05,3.45
0,desconocido,Otro/Desconocido,3423,107,3.318389e+04,3.13
3,desktop,MacOS,13572,298,1.717912e+05,2.20
1,desktop,Android,1,0,0.000000e+00,0.00
